# Module 12 - 실습 4 솔루션: astream 토큰 스트리밍

In [ ]:
import sys
sys.path.insert(0, '../..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("스트리밍 솔루션 준비 완료")

In [ ]:
class StreamState(TypedDict):
    query: str
    research: str | None
    analysis: str | None
    final_answer: str | None

fake_llm = FakeLLM(responses={
    "default": "분석 결과: Python 비동기 프로그래밍은 asyncio 라이브러리를 사용합니다."
})

def research_node(state: StreamState) -> dict:
    """리서치를 수행합니다."""
    query = state["query"]
    research = f"[리서치] '{query}'에 대한 자료를 수집했습니다. Python asyncio는 코루틴 기반 비동기 IO를 지원합니다."
    print(f"  [research] 완료")
    return {"research": research}

def analyze_node(state: StreamState) -> dict:
    """LLM으로 분석을 수행합니다."""
    research = state.get("research", "")
    response = fake_llm.invoke(research)
    analysis = response.content if hasattr(response, 'content') else str(response)
    print(f"  [analyze] 완료")
    return {"analysis": analysis}

def answer_node(state: StreamState) -> dict:
    """최종 답변을 생성합니다."""
    analysis = state.get("analysis", "")
    final_answer = f"최종 답변: {analysis}"
    print(f"  [answer] 완료")
    return {"final_answer": final_answer}

# 그래프 구성
graph = StateGraph(StreamState)
graph.add_node("research", research_node)
graph.add_node("analyze", analyze_node)
graph.add_node("answer", answer_node)

graph.set_entry_point("research")
graph.add_edge("research", "analyze")
graph.add_edge("analyze", "answer")
graph.add_edge("answer", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)
print("그래프 구성 완료")

In [ ]:
# updates 모드 스트리밍
config = {"configurable": {"thread_id": "stream-001"}}
input_state = {
    "query": "Python 비동기 프로그래밍의 장단점",
    "research": None,
    "analysis": None,
    "final_answer": None,
}

print("=== 노드 업데이트 스트리밍 ===\n")
received_nodes = []

async for event in app.astream(input_state, config=config, stream_mode="updates"):
    for node_name, updates in event.items():
        received_nodes.append(node_name)
        print(f"[{node_name}] 완료")
        for key, value in updates.items():
            if value is not None:
                preview = str(value)[:60] + "..." if len(str(value)) > 60 else str(value)
                print(f"  {key}: {preview}")

print(f"\n수신한 노드 순서: {received_nodes}")

In [ ]:
# 검증
assert len(received_nodes) >= 3, f"최소 3개 노드 업데이트를 수신해야 합니다. 현재: {received_nodes}"
assert "research" in received_nodes, "research 노드 이벤트를 수신해야 합니다"
assert "analyze" in received_nodes, "analyze 노드 이벤트를 수신해야 합니다"
assert "answer" in received_nodes, "answer 노드 이벤트를 수신해야 합니다"
print("✅ 실습 2 완료! 노드 단위 스트리밍이 올바르게 작동합니다.")

In [ ]:
# values 모드 스트리밍
config2 = {"configurable": {"thread_id": "stream-002"}}
input_state2 = {
    "query": "asyncio vs threading 비교",
    "research": None,
    "analysis": None,
    "final_answer": None,
}

print("=== values 모드 스트리밍 ===\n")
state_snapshots = []

async for state in app.astream(input_state2, config=config2, stream_mode="values"):
    state_snapshots.append(dict(state))
    print(f"현재 상태: research={state.get('research') is not None}, "
          f"analysis={state.get('analysis') is not None}, "
          f"final_answer={state.get('final_answer') is not None}")

print(f"\n총 {len(state_snapshots)}번 상태 업데이트 수신")

In [ ]:
# 검증
assert len(state_snapshots) >= 3, f"최소 3번의 상태 업데이트가 있어야 합니다. 현재: {len(state_snapshots)}"
final_state = state_snapshots[-1]
assert final_state.get("research") is not None, "최종 상태에 research가 없습니다"
assert final_state.get("analysis") is not None, "최종 상태에 analysis가 없습니다"
assert final_state.get("final_answer") is not None, "최종 상태에 final_answer가 없습니다"
print("✅ 실습 3 완료! values 모드 스트리밍이 올바르게 작동합니다.")